In [61]:
import os
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import pertpy

# Check current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/pedroferreira/projects/DE-ZILN/notebooks/scrnaseq_reanalyses


In [62]:
adata = pertpy.data.kang_2018()
print(adata)

AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'


In [63]:

# -----------------------
# Step 2: Inspect metadata
# -----------------------
# 'label' is the column that contains "ctrl" or "stim"
print("Unique condition labels in obs:", adata.obs["label"].unique())

# Rename it for clarity
adata.obs["condition"] = adata.obs["label"].astype("category")

# -----------------------------
# Step 3: Basic scanpy preprocessing
# -----------------------------
# Store raw counts for later reference
adata.layers["counts"] = adata.X.copy()

# Take only T-cells
adata = adata[adata.obs['cell_type'].str.contains('T cells')].copy()
print(adata.obs['cell_type'].value_counts())

# QC filters (these are typical thresholds; adjust if needed)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# Mitochondrial content
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()

# Remove doublets
sc.pp.scrublet(adata)
adata = adata[adata.obs["predicted_doublet"] == False].copy()

# Normalize + log1p
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

# ------------------------
# Step 4: Run DGE (t-test)
# ------------------------
sc.tl.rank_genes_groups(
    adata,
    groupby="condition",
    groups=["stim"],
    reference="ctrl",
    method="t-test"
)

print("Top 20 DGE results:")
df = sc.get.rank_genes_groups_df(adata, group="stim")
print(df.head(20))



Unique condition labels in obs: ['ctrl', 'stim']
Categories (2, object): ['ctrl', 'stim']
cell_type
CD4 T cells    11238
CD8 T cells     1621
Name: count, dtype: int64
Top 20 DGE results:
      names      scores  logfoldchanges  pvals  pvals_adj
0     ISG15  192.520645       10.905020    0.0        0.0
1      IFI6  179.198242       11.142301    0.0        0.0
2     IFIT3  152.896469       11.938686    0.0        0.0
3     IFIT1  141.387939       11.451269    0.0        0.0
4       MX1  113.908150        8.966833    0.0        0.0
5      LY6E  112.923683        8.427493    0.0        0.0
6     ISG20  100.401672        7.316315    0.0        0.0
7     IFIT2   87.951065        9.176761    0.0        0.0
8      OAS1   83.281662        8.123415    0.0        0.0
9      MT2A   77.013077        7.083405    0.0        0.0
10   IFI44L   72.969772        7.479745    0.0        0.0
11    RSAD2   72.616829        8.724595    0.0        0.0
12     IRF7   66.330666        6.037930    0.0        0.0


In [ ]:
# -------------------------------
# Negative control: Split control condition into two random groups
# -------------------------------
# Filter to only control cells
ctrl_adata = adata[adata.obs["condition"] == "ctrl"].copy()
ctrl_adata.X = ctrl_adata.layers['counts'].copy()
sc.pp.filter_genes(ctrl_adata, min_cells=3)
sc.pp.normalize_total(ctrl_adata, target_sum=1e6)
sc.pp.log1p(ctrl_adata)

print(f"Total control cells: {ctrl_adata.n_obs}")

rng = np.random.default_rng(1)

reps = np.array(sorted(ctrl_adata.obs["replicate"].unique()))
rng.shuffle(reps)

# Split replicates roughly in half
half = len(reps) // 2
groupA = set(reps[:half])
groupB = set(reps[half:])

# Assign each cell based on replicate membership
grp = ctrl_adata.obs["replicate"].map(lambda r: "A" if r in groupA else "B")
ctrl_adata.obs["rand_group"] = grp.astype("category")

# Sanity check: replicate counts per group
rep_counts = ctrl_adata.obs.drop_duplicates("replicate")["rand_group"].value_counts()
print("Replicates per random group:", rep_counts.to_dict())

# Cell counts per group
print("Cells per random group:", ctrl_adata.obs["rand_group"].value_counts().to_dict())

# Run cell-level DE ignoring replicate structure (this is the point)
sc.tl.rank_genes_groups(
    ctrl_adata,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    method='t-test',
    use_raw=False
)
results_df = sc.get.rank_genes_groups_df(ctrl_adata, group="A")


# Count significant genes at different thresholds, considering only abs(LFC) > 0.5
alpha = 0.05
lfc_mask = results_df["logfoldchanges"].abs() > 0.5
n_de_05 = ((results_df["pvals_adj"] < alpha) & lfc_mask).sum()
n_de_01 = ((results_df["pvals_adj"] < 0.01) & lfc_mask).sum()
n_de_001 = ((results_df["pvals_adj"] < 0.001) & lfc_mask).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Total control cells: 6371
Replicates per random group: {'A': 4, 'B': 4}
Cells per random group: {'B': 3628, 'A': 2743}

Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 228
DE genes (FDR < 0.01): 155
DE genes (FDR < 0.001): 96

False positive rate (FDR < 0.05): 1.81%

Top 10 'differentially expressed' genes (should be false positives):
    names         pvals     pvals_adj     scores  logfoldchanges
0   HLA-C  1.883162e-46  7.896099e-43  14.427890        1.108141
1    FTH1  2.552676e-36  6.422021e-33  12.668288        0.635306
2   RPS26  7.135995e-25  9.973742e-22  10.346748        1.411763
3    CREM  1.996025e-24  2.510800e-21  10.246393        1.410902
4   GAPDH  4.843350e-22  5.077041e-19   9.689387        1.312225
5    IL7R  2.194381e-18  2.123317e-15   8.776340        1.178745
6    PLP2  3.043079e-17  2.734207e-14   8.472615        1.136303
7  GPR183  1.584246e-15  1.172249e-12   7.992980        1.090281
8   TRAT1  6.182252e-15  4.0929

In [65]:
# Run cell-level DE ignoring replicate structure (this is the point)
sc.tl.rank_genes_groups(
    ctrl_adata,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    method='wilcoxon',
    use_raw=False
)
results_df = sc.get.rank_genes_groups_df(ctrl_adata, group="A")


# Count significant genes at different thresholds, considering only abs(LFC) > 0.5
alpha = 0.05
lfc_mask = results_df["logfoldchanges"].abs() > 0.5
n_de_05 = ((results_df["pvals_adj"] < alpha) & lfc_mask).sum()
n_de_01 = ((results_df["pvals_adj"] < 0.01) & lfc_mask).sum()
n_de_001 = ((results_df["pvals_adj"] < 0.001) & lfc_mask).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 50
DE genes (FDR < 0.01): 45
DE genes (FDR < 0.001): 31

False positive rate (FDR < 0.05): 0.40%

Top 10 'differentially expressed' genes (should be false positives):
     names         pvals     pvals_adj     scores  logfoldchanges
0    HLA-C  1.112589e-70  1.399526e-66  17.774565        1.108141
1     FTH1  6.837187e-49  4.300249e-45  14.696010        0.635306
2    RPS26  1.337916e-27  4.207411e-24  10.886424        1.411763
3     CREM  1.421359e-22  2.979879e-19   9.776430        1.410902
4    GAPDH  2.350512e-18  3.285232e-15   8.739063        1.312225
5     IL7R  1.063441e-14  1.337702e-11   7.731431        1.178745
6    RPLP1  1.413385e-13  1.367613e-10   7.395066        0.206209
7    RPS16  2.379622e-13  2.138090e-10   7.325527        0.398020
8   GPR183  2.928821e-11  2.270890e-08   6.650106        1.090281
9  S100A11  3.069014e-11  2.270890e-08   6.643221        0.966330


In [66]:
import importlib
import os

file_path = os.path.join(os.getcwd(), "../../pkg/", "scanpy_wrapper.py")
spec = importlib.util.spec_from_file_location("scanpy_wrapper", file_path)
scanpy_wrapper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scanpy_wrapper)

# LN's t-test uses normalized data without log1p
ctrl_adata_norm = ctrl_adata.copy()
ctrl_adata_norm.X = ctrl_adata_norm.layers['counts'].copy()
sc.pp.normalize_total(ctrl_adata_norm, target_sum=1e6)
scanpy_wrapper.rank_genes_groups_ln(
    ctrl_adata_norm,
    groupby="rand_group",
    groups=["A"],
    reference="B",
    use_raw=False
)

Processing group: A


In [67]:
results_df = sc.get.rank_genes_groups_df(ctrl_adata_norm, group="A")


# Count significant genes at different thresholds, considering only abs(LFC) > 0.5
alpha = 0.05
lfc_mask = results_df["logfoldchanges"].abs() > 0.5
n_de_05 = ((results_df["pvals_adj"] < alpha) & lfc_mask).sum()
n_de_01 = ((results_df["pvals_adj"] < 0.01) & lfc_mask).sum()
n_de_001 = ((results_df["pvals_adj"] < 0.001) & lfc_mask).sum()

print("\n" + "="*60)
print("Negative Control Results (Control vs Control)")
print("="*60)
print(f"Total genes tested: {len(results_df)}")
print(f"DE genes (FDR < 0.05): {n_de_05}")
print(f"DE genes (FDR < 0.01): {n_de_01}")
print(f"DE genes (FDR < 0.001): {n_de_001}")
print(f"\nFalse positive rate (FDR < 0.05): {n_de_05 / len(results_df) * 100:.2f}%")

print("\nTop 10 'differentially expressed' genes (should be false positives):")
print(results_df.head(10)[["names", "pvals", "pvals_adj", "scores", "logfoldchanges"]])


Negative Control Results (Control vs Control)
Total genes tested: 12579
DE genes (FDR < 0.05): 50
DE genes (FDR < 0.01): 42
DE genes (FDR < 0.001): 32

False positive rate (FDR < 0.05): 0.40%

Top 10 'differentially expressed' genes (should be false positives):
     names         pvals     pvals_adj     scores  logfoldchanges
0     FTH1  0.000000e+00  0.000000e+00  15.982588        0.596797
1    HLA-C  0.000000e+00  0.000000e+00  15.780686        0.413480
2    RPS26  1.401298e-45  1.271678e-41  14.308333        0.717218
3     CREM  1.199717e-28  1.509125e-24  11.159313        0.673358
4    RPS16  9.677427e-16  1.217323e-11   8.052910        0.209996
5   CLDND1  3.019346e-15  3.798034e-11   7.909792        0.832453
6      VIM  2.134449e-14  2.684924e-10   7.661064        0.322621
7    RPLP1  5.660303e-14  7.120095e-10   7.533543        0.130438
8     IL7R  1.543610e-13  1.941706e-09   7.399652        0.427414
9  S100A11  6.547625e-13  8.236258e-09   7.203787        0.380071
